In [1]:
import ee
import folium
from IPython.display import Image, display

try:
    ee.Initialize(project="ee-undergrad-thesis-509519")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="ee-undergrad-thesis-509519")
    
WEST, SOUTH, EAST, NORTH = 37.48, 47.05, 37.68, 47.15
ROI = ee.Geometry.BBox(WEST, SOUTH, EAST, NORTH)

PRE_START,  PRE_END  = "2021-05-15", "2021-07-15"   # pre-war, summer 2021
POST_START, POST_END = "2022-06-01", "2022-08-01"   # post-siege, summer 2022
 
BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
CRS   = "EPSG:32637"   # UTM zone 37N (covers 36-42 E)
SCALE = 10             # metres
S2    = "COPERNICUS/S2_SR_HARMONIZED"

In [2]:
m = folium.Map(location=[(SOUTH + NORTH) / 2, (WEST + EAST) / 2], zoom_start=13)
folium.Rectangle(bounds=[[SOUTH, WEST], [NORTH, EAST]], color="red", fill=False).add_to(m)
m 

In [3]:
BAD_SCL = [0, 1, 3, 8, 9, 10, 11]
 
def mask_s2_clouds(img):
    scl = img.select("SCL")
    bad = ee.Image(0)
    for c in BAD_SCL:
        bad = bad.Or(scl.eq(c))
    return img.updateMask(bad.Not())

In [4]:
test_col = (
    ee.ImageCollection(S2)
    .filterBounds(ROI)
    .filterDate(PRE_START, POST_END)
    .filter(ee.Filter.And(
        ee.Filter.gt("CLOUDY_PIXEL_PERCENTAGE", 15),
        ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60),
    ))
    .sort("CLOUDY_PIXEL_PERCENTAGE", False)
)
raw = ee.Image(test_col.first())
masked = mask_s2_clouds(raw)

date = ee.Date(raw.get("system:time_start")).format("YYYY-MM-dd").getInfo()
cloud = raw.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()
print(f"Test image date: {date} | scene cloud %: {cloud:.1f}")
 
# fraction of the ROI removed by the mask (SCL is 20 m)
kept = masked.select("B4").mask().reduceRegion(ee.Reducer.mean(), ROI, 20).getInfo()["B4"]
print(f"Fraction of ROI kept after masking: {kept:.2f}")
 
vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
thumb = {"region": ROI, "dimensions": 500, "format": "png"}
print("BEFORE")
display(Image(url=raw.getThumbURL({**vis, **thumb})))
print("AFTER (masked pixels shown black)")
display(Image(url=masked.unmask(0).getThumbURL({**vis, **thumb})))

Test image date: 2022-04-03 | scene cloud %: 57.2
Fraction of ROI kept after masking: 0.40
BEFORE


AFTER (masked pixels shown black)


In [5]:
for name, (s, e) in {"pre": (PRE_START, PRE_END), "post": (POST_START, POST_END)}.items():
    n = ee.ImageCollection(S2).filterBounds(ROI).filterDate(s,e).size().getInfo()
    print(f"{name}: {n} scenes between {s} and {e}")

pre: 12 scenes between 2021-05-15 and 2021-07-15
post: 12 scenes between 2022-06-01 and 2022-08-01


In [6]:
def make_composite(start, end):
    col = (
        ee.ImageCollection(S2)
        .filterBounds(ROI)
        .filterDate(start, end)
        .map(mask_s2_clouds)
    )
    return col.select(BANDS).median().clip(ROI)

pre = make_composite(PRE_START, PRE_END)
post = make_composite(POST_START, POST_END)

In [7]:
for name, img in {"pre": pre, "post": post}.items():
    valid = img.select("B4").mask().reduceRegion(ee.Reducer.mean(), ROI, 20).getInfo()["B4"]
    print(f"{name}: {valid:.1%} of the ROI has valid pixels")

pre: 99.9% of the ROI has valid pixels
post: 99.9% of the ROI has valid pixels
